# M39 — Add Memory, Routing, and Fallbacks

M38 already gave you an explicit state machine around tools. The useful whole
here is **robustness**: persistent memory with provenance, explicit routes,
and a fallback ladder that can stop.

This is not another M38 node. It is not an M40 eval harness. It is the last
V10 step: make the agent fail closed instead of looping or silently using a
stale catalog fact.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a `route` name, a retrieved id, an `attempts`
count, a `degraded` flag, a `posted_amount`, or a terminal.

Do not hide routing in a prompt. Do not treat a fluent transcript as
permission to post an expired price. The repository does not prefill learner
answers, ADR text, or competence.

Canonical sources (named, not imported): `langgraph-docs`, `anthropic-agents`.
Content bundle: `tool-using-agents` (M37+M38+M39).


In [ ]:
from pathlib import Path
import inspect
import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M39" / "robust_agent.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M38.agent_workflow import (
    CATALOG_PRICE as M38_CATALOG_PRICE,
    CATALOG_SKU as M38_CATALOG_SKU,
    STATE_FIELDS as M38_STATE_FIELDS,
)
from missions.M39.robust_agent import (
    CATALOG_PRICE,
    CATALOG_SKU,
    CIRCUIT_THRESHOLD,
    DEFAULT_NOW,
    DEFAULT_TASK,
    FIELD_CLASSIFICATION,
    HANDOFF,
    LADDER,
    LOOKUP_TASK,
    MAX_ATTEMPTS,
    MEM_SKU7_FRESH,
    MEM_SKU7_PRICE,
    MEM_SKU7_STALE,
    NO_MATCH_TASK,
    PERSISTENT_CANDIDATES,
    ROUTE_NAMES,
    ROBUST_VERSION,
    SCALE_LIMIT,
    SEED,
    SYSTEM_MAP,
    TRACE_FIELDS,
    WORKING_EPHEMERAL,
    LiveAdapterUnavailable,
    MemoryStore,
    OptionalLangGraphUnavailable,
    catalog_price_entry,
    demo_store_relevant_and_irrelevant,
    demo_store_stale_and_fresh,
    demo_store_superseded,
    graph_public,
    handoff_contract,
    memory_query_from_task,
    observability_report,
    optional_langgraph_store,
    optional_live_retrieve,
    pipeline_with_defect,
    repair_run,
    retrieve_memory,
    run_fallback_ladder,
    run_robust_task,
    select_route,
)

print("repository root:", ROOT)
print("robust version:", ROBUST_VERSION)
print("seed:", SEED)
print("scale:\n", SCALE_LIMIT)


## M38 → M39 boundary

M38's `AgentState` already carries `node`, `last_tool_result`, `approval`,
and `history`. Those fields are **working memory**: they describe the current
run. They are a bad place to keep a catalog fact you might need next week.

LangGraph's docs split the same idea as short-term (thread/checkpoint) versus
long-term (store) memory. Anthropic's agent guidance treats routing as an
explicit workflow: classify the input, then send it down a specialized path.
This mission implements those ideas as local fixtures wrapping
`missions.M38.agent_workflow`. No SDK is imported.

Start by classifying the inherited fields. Then run the smallest useful robust
task: retrieve → route → wrap M38 → fallback only if the primary fails.


In [ ]:
print("field classification:")
print(FIELD_CLASSIFICATION)
print("working ephemeral:", WORKING_EPHEMERAL)
print("persistent candidates:", PERSISTENT_CANDIDATES)
print("m38 state fields:", M38_STATE_FIELDS)
print("system map:\n", SYSTEM_MAP)
print("graph:")
print(json.dumps(graph_public(), indent=2))
print("route names:", ROUTE_NAMES)
print("ladder:", LADDER)
print("max_attempts:", MAX_ATTEMPTS)
print("circuit_threshold:", CIRCUIT_THRESHOLD)
print("trace fields:", TRACE_FIELDS)
print("default now:", DEFAULT_NOW)
print("catalog sku constant:", CATALOG_SKU)
print("m38 catalog sku:", M38_CATALOG_SKU)

try:
    optional_live_retrieve("SKU-7")
except LiveAdapterUnavailable as exc:
    print("live adapter:", type(exc).__name__)
try:
    optional_langgraph_store()
except OptionalLangGraphUnavailable as exc:
    print("langgraph adapter:", type(exc).__name__)


The map is the whole machine: memory, router, M38 wrap, ladder, terminals.
`FIELD_CLASSIFICATION` is not optional flavor text. If you dump `history` into
a durable store, you have mixed working state with facts. If you skip
provenance, you cannot later tell a stale price from a current one.

Live and LangGraph adapters fail closed. The required path stays local.


## Predict before running — whole robust task

Timestamp a prediction before `run-whole`.

Fixed task: purchase SKU-7 and post the catalog price. The store currently
holds one catalog-price fact for that SKU. Local fixtures, not a live model.
Approval is granted.

Predict:

- the route name
- the terminal
- whether `degraded` is true
- whether `attempts` is 1 or more than 1
- whether a fallback rung runs


In [ ]:
print(SYSTEM_MAP)
whole_store = MemoryStore().put(catalog_price_entry())
whole = run_robust_task(DEFAULT_TASK, store=whole_store)
print("route", whole.route)
print("terminal", whole.terminal)
print("degraded", whole.degraded)
print("attempts", whole.attempts)
print("retrieved_ids", whole.retrieved_ids)
print("used_memory_ids", whole.used_memory_ids)
print("fallbacks_used", whole.fallbacks_used)
print("posted_amount", whole.posted_amount)
print("effect_count", whole.effect_count)
print("executions", whole.workflow.session.executions)
print("circuit_open", whole.circuit_open)
print("weights_updated", whole.workflow.state.inference["weights_updated"])
print("m38 module", type(whole.workflow.state).__module__)
print("handoff\n", HANDOFF)


A healthy purchase can retrieve a relevant fact, choose `catalog_purchase`,
and wrap M38 without climbing the ladder. Retrieved ids are context.
`used_memory_ids` stays empty when the catalog lookup is still the source of
truth. That distinction matters the moment memory goes stale.

This run is not a production agent. It is the useful whole at teaching scale.


## Predict before running — memory relevance

Timestamp a prediction before `run-relevance`.

One named change: the same purchase task, but the store now also contains a
warehouse quantity row and a price row for a different SKU. Retrieval policy
is frozen. The task is frozen.

Predict:

- which entry ids are in `retrieved_ids`
- which ids are excluded, and why (scope versus sku)
- whether the posted path still wraps M38 instead of trusting a warehouse qty


In [ ]:
relevance_store = demo_store_relevant_and_irrelevant()
print("store ids", [entry.entry_id for entry in relevance_store.entries])
print(
    "store keys/scopes",
    [(entry.entry_id, entry.key, entry.scope, entry.sku) for entry in relevance_store.entries],
)
relevance = run_robust_task(DEFAULT_TASK, store=relevance_store)
print("retrieved_ids", relevance.retrieved_ids)
print("excluded", relevance.excluded)
print("used_memory_ids", relevance.used_memory_ids)
print("route", relevance.route)
print("posted_amount", relevance.posted_amount)
print("effect_count", relevance.effect_count)


Irrelevant rows are not "low ranked." They are out of scope or the wrong SKU.
A store dump is not retrieval. If the warehouse quantity had been appended to
the prompt, a model could have treated it as evidence. The policy never gave
it a chance.


## Predict before running — stale memory

Timestamp a prediction before `run-stale`.

One named change: the query stays a SKU-7 purchase, `now=1000`, and the store
has two price rows for that SKU.

- `mem-sku7-price-stale`: `written_at=100`, `expires_at=200`
- `mem-sku7-price-fresh`: `written_at=800`, `expires_at=2000`

Predict:

- which ids appear in `retrieved_ids`
- which ids are flagged or excluded, and the reason string
- whether `used_memory_ids` is empty
- whether the posted amount comes from the wrapped M38 lookup


In [ ]:
stale_store = demo_store_stale_and_fresh()
print("ids", [entry.entry_id for entry in stale_store.entries])
print(
    "times",
    [
        (entry.entry_id, entry.provenance.written_at, entry.expires_at)
        for entry in stale_store.entries
    ],
)
print("now", DEFAULT_NOW)
stale = run_robust_task(DEFAULT_TASK, store=stale_store, now=DEFAULT_NOW)
print("retrieved_ids", stale.retrieved_ids)
print("excluded", stale.excluded)
print("used_memory_ids", stale.used_memory_ids)
print("posted_amount", stale.posted_amount)
print("degraded", stale.degraded)
print("terminal", stale.terminal)


Expiry is a lifecycle field, not a hint in a prompt. An expired row can stay
in the store for audit and still be unusable as a price. Later, a named defect
will skip this check on purpose. This cell did not skip it.


## Predict before running — route selection

Timestamp a prediction before `run-routes`.

Router policy is frozen. Apply it to this fixed case set:

1. Purchase SKU-7 and post the catalog price to the cash ledger.
2. Look up the catalog price of SKU-7.
3. Compose a haiku about warehouse bins.

Allowed names: `catalog_purchase`, `catalog_lookup`, `no_match`.
Precedence is purchase before lookup. No-match must not wrap M38.

Predict:

- the route for each case
- whether case 3 runs `run_workflow`
- `effect_count` for the lookup case versus the purchase case


In [ ]:
routed = {}
for task in (DEFAULT_TASK, LOOKUP_TASK, NO_MATCH_TASK):
    decision = select_route(task)
    result = run_robust_task(task, store=whole_store)
    routed[task] = result
    print("task:", task)
    print("  route", decision.route)
    print("  reason", decision.reason)
    print("  terminal", result.terminal)
    print("  wrapped_m38", result.workflow is not None)
    print("  effect_count", result.effect_count)
    print("  attempts", result.attempts)
lookup_run = routed[LOOKUP_TASK]
refused = routed[NO_MATCH_TASK]


Routing is a predicate, not a vibe. Anthropic describes this as classifying an
input and directing it to a specialized follow-up. A haiku is not a catalog
tool call. Refusing is safer than inventing a purchase. Lookup completes
without posting; that is success for that route, not a degraded purchase.


## Predict before running — primary failure

Timestamp a prediction before `run-primary-failure`.

One named change: the purchase task is fixed, and the primary rung is forced
to fail. The ladder and catalog stay frozen.

Predict:

- whether a fallback rung runs
- `attempts`
- whether `degraded` is true
- whether `terminal` is `complete` or something else
- whether `effect_count` stays 0


In [ ]:
primary_fail = run_robust_task(DEFAULT_TASK, store=whole_store, inject="primary_failure")
print("route", primary_fail.route)
print("terminal", primary_fail.terminal)
print("degraded", primary_fail.degraded)
print("attempts", primary_fail.attempts)
print("fallbacks_used", primary_fail.fallbacks_used)
print("circuit_open", primary_fail.circuit_open)
print("effect_count", primary_fail.effect_count)
print("claim", primary_fail.claim)
print("last_tool_result", primary_fail.last_tool_result)
print("trace", primary_fail.trace)


Degraded success is a labeled terminal. The fallback still wrapped M38, but
only through lookup. That is not silent incorrect success: the ledger was not
posted, and `degraded` is true. A system that marked this `complete` would be
lying.


## Predict before running — fallback loop

Timestamp a prediction before `run-loop`.

One named change: primary and fallback both fail. Policies stay frozen.
`max_attempts` and `circuit_threshold` are the ones printed in the map.

Predict:

- `attempts`
- whether `circuit_open` is true
- the terminal name
- whether the run can continue past the bound


In [ ]:
looped = run_robust_task(DEFAULT_TASK, store=whole_store, inject="all_failures")
print("terminal", looped.terminal)
print("degraded", looped.degraded)
print("attempts", looped.attempts)
print("max_attempts", MAX_ATTEMPTS)
print("circuit_threshold", CIRCUIT_THRESHOLD)
print("circuit_open", looped.circuit_open)
print("circuit_failures", looped.circuit_failures)
print("aborted_ceiling", looped.aborted_ceiling)
print("rungs", [event["rung"] for event in looped.trace])
print("effect_count", looped.effect_count)


A circuit can open before the attempt cap is exhausted. That is the point of
having both knobs. Unbounded retry is not robustness. Later, a named defect
will ignore both knobs. This cell did not.


In [ ]:
labels = ["happy path", "primary fail", "both fail", "no-match"]
plot_results = [whole, primary_fail, looped, refused]
terminals = [item.terminal for item in plot_results]
attempts = [item.attempts for item in plot_results]
degraded = [item.degraded for item in plot_results]
rank = {"complete": 4, "degraded": 3, "circuit_open": 2, "no_match": 1, "failed": 0}
heights = [rank.get(item, 0) for item in terminals]
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#2a9d8f", "#e9c46a", "#e76f51", "#6d6875"]
ax.bar(np.arange(len(labels)), heights, color=colors)
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticks([1, 2, 3, 4])
ax.set_yticklabels(["no_match", "circuit_open", "degraded", "complete"])
ax.set_ylabel("terminal rank")
ax.set_xlabel("one named change from the SKU-7 robust purchase")
ax.set_title("Which run completed, degraded, opened a circuit, or refused?")
ax.set_ylim(0, 4.5)
fig.tight_layout()
plt.show()
print("plot asks: which run completed vs degraded vs circuit_open vs no_match?")
print("terminals", list(zip(labels, terminals)))
print("attempts", list(zip(labels, attempts)))
print("degraded_flags", list(zip(labels, degraded)))


## Predict before running — code reading

Timestamp a prediction before `run-code-reading`.

Read `retrieve_memory`, `select_route`, `run_fallback_ladder`,
`run_robust_task`, and `repair_run` in `missions/M39/robust_agent.py`.
Dump `inspect.getsource` of those functions and probe live objects
(`retrieved_ids`, `route`, `attempts`, `degraded`).

Predict:

- whether `retrieve_memory` puts an expired SKU-7 price in `included_ids`
- what `select_route` returns for a haiku with no SKU
- what `run_fallback_ladder` records for `attempts` and `circuit_open` when
  primary and fallback both fail
- whether `run_robust_task` wraps M38 on `no_match`
- whether a lookup-only fallback of a purchase is `degraded` or `complete`
- what `repair_run` reuses from a broken object


In [ ]:
print("### retrieve_memory")
print(inspect.getsource(retrieve_memory))
print("### select_route")
print(inspect.getsource(select_route))
print("### run_fallback_ladder")
print(inspect.getsource(run_fallback_ladder))
print("### run_robust_task")
print(inspect.getsource(run_robust_task))
print("### repair_run")
print(inspect.getsource(repair_run))

print("whole retrieved_ids", whole.retrieved_ids)
print("whole route", whole.route)
print("whole attempts", whole.attempts)
print("whole degraded", whole.degraded)
print("relevance retrieved_ids", relevance.retrieved_ids)
print("stale retrieved_ids", stale.retrieved_ids)
print("stale excluded", stale.excluded)
print("primary_fail attempts", primary_fail.attempts)
print("primary_fail degraded", primary_fail.degraded)
print("primary_fail terminal", primary_fail.terminal)
print("looped attempts", looped.attempts)
print("looped circuit_open", looped.circuit_open)
print("refused route", refused.route)
print("haiku live route", select_route(NO_MATCH_TASK).route)

super_store = demo_store_superseded()
super_query = memory_query_from_task(DEFAULT_TASK, route="catalog_purchase")
super_ret = retrieve_memory(super_store, super_query, now=DEFAULT_NOW)
print("supersede included_ids", super_ret.included_ids)
print("supersede excluded ids", [item.entry_id for item, _reason in super_ret.excluded])
print("supersede reasons", [reason for _item, reason in super_ret.excluded])
report = observability_report(whole)
print("handoff", report["handoff"])
print("trace fields", report["trace_fields"])
print("weights_updated on live state", whole.workflow.state.inference["weights_updated"])


## Predict before running — Controlled failure: expired memory treated as truth

Timestamp a prediction before `run-failure`.

The store still has the two SKU-7 prices from the stale experiment. A broken
path skips the lifecycle check and treats a remembered amount as a current
catalog price, then marks the run complete.

Predict, from provenance/expiry traces you would inspect:

- whether `degraded` is true
- whether lookup still runs
- whether `posted_amount` matches the catalog fixture or the expired row
- which memory id, if any, is in `used_memory_ids`


In [ ]:
broken_stale = pipeline_with_defect(defect="stale_memory_trusted")
print("defect", broken_stale.defect)
print("terminal", broken_stale.terminal)
print("degraded", broken_stale.degraded)
print("posted_amount", broken_stale.posted_amount)
print("used_memory_ids", broken_stale.used_memory_ids)
print("retrieved_ids", broken_stale.result.retrieved_ids)
print("excluded", broken_stale.result.excluded)
print("executions", broken_stale.audit["executions"])
print("claim", broken_stale.claim)
print("catalog_price in audit", broken_stale.audit["catalog_price"])
print("stale_price in audit", broken_stale.audit["stale_price"])


## Predict before running — Controlled failure: fallbacks that never stop

Timestamp a prediction before `run-failure-oscillation`.

A second named defect, not a simultaneous repair of the first. Primary and
fallback keep failing. The broken path ignores the circuit and the attempt
bound.

Predict, from route/attempt traces you would inspect:

- whether `circuit_open` is true
- whether `attempts` stays at `CIRCUIT_THRESHOLD` or grows past `MAX_ATTEMPTS`
- whether `aborted_ceiling` is true
- the rung pattern in the trace


In [ ]:
broken_osc = pipeline_with_defect(defect="fallback_oscillation")
print("defect", broken_osc.defect)
print("terminal", broken_osc.terminal)
print("attempts", broken_osc.attempts)
print("max_attempts", MAX_ATTEMPTS)
print("circuit_threshold", CIRCUIT_THRESHOLD)
print("circuit_open", broken_osc.circuit_open)
print("aborted_ceiling", broken_osc.result.aborted_ceiling)
print("rungs", broken_osc.audit["rungs"])
print("claim", broken_osc.claim)


## Diagnose from provenance, expiry, route, and attempt traces

Read the two broken objects before you repair anything.

For the first, compare `posted_amount` to the catalog fixture, then read
`used_memory_ids`, `retrieved_ids`, executions, and `expires_at` versus `now`.
Ask whether `complete` with `degraded=False` is honest.

For the second, compare `attempts` to `MAX_ATTEMPTS`, then read `circuit_open`
and the rung list. Ask whether the ladder is a bound or a while-true.

Do not start with a larger model. Do not import LangGraph. Repair each named
defect in its own predict/act step from the **broken object**.


## Predict before running — repair the stale-memory path

Timestamp a prediction before `run-failure-repair`.

Call `repair_run` on the stale-memory broken object only. One named change.
The oscillation object stays broken until a later cell.

Predict:

- the repaired `posted_amount`
- whether `used_memory_ids` is empty
- whether the original object still shows the stale complete


In [ ]:
repaired_stale = repair_run(broken_stale)
print("repaired terminal", repaired_stale.terminal)
print("repaired degraded", repaired_stale.degraded)
print("repaired posted_amount", repaired_stale.posted_amount)
print("repaired used_memory_ids", repaired_stale.used_memory_ids)
print("repaired retrieved_ids", repaired_stale.result.retrieved_ids)
print("repaired executions", repaired_stale.result.workflow.session.executions)
print("broken still posted", broken_stale.posted_amount)
print("broken still terminal", broken_stale.terminal)
print("broken still used", broken_stale.used_memory_ids)


The repair must reuse the broken object's initial store and task. A second
unrelated happy-path run from module defaults is not a repair. The original
object remains a regression: it still posts the expired amount as complete.


## Predict before running — repair the unbounded fallback

Timestamp a prediction before `run-oscillation-repair`.

Call `repair_run` on the oscillation broken object only. One named change.
Do not re-repair the stale-memory object here.

Predict:

- repaired `attempts`
- whether `circuit_open` is true
- whether the original object still exceeds `MAX_ATTEMPTS`


In [ ]:
repaired_osc = repair_run(broken_osc)
print("repaired terminal", repaired_osc.terminal)
print("repaired attempts", repaired_osc.attempts)
print("repaired circuit_open", repaired_osc.circuit_open)
print("repaired aborted", repaired_osc.result.aborted_ceiling)
print("broken still attempts", broken_osc.attempts)
print("broken still circuit_open", broken_osc.circuit_open)
print("broken still aborted", broken_osc.result.aborted_ceiling)


The bound is orchestration policy, not a suggestion in a prompt. After repair,
the same injected failures stop at the circuit. The broken object still
oscillates. Keep both objects; M40 will want traces like these, not a green
dashboard with no failures.


## Evidence contract

Submit timestamped predictions, the field-classification map, retrieval
exclusions, route outcomes including no-match, a degraded primary-failure
run, a circuit-bound run, both named-defect diagnoses, both repairs from
broken objects, and regression evidence that the broken paths still fail.

A running notebook is not competence. [UNFILLED BY LEARNER]


## No-AI gate

Complete `missions/M39/no_ai_gate.md` from a blank page. The fixture uses
SKU-21, BIN-8, and order 4401 — not this notebook's SKU-7 purchase.

Classify workflow state versus persistent memory, design provenance/expiry
fields, route three fresh cases, draw a two-level fallback with a hard stop,
and diagnose one teammate trace.

Leave all learner responses unfilled in the repository. [UNFILLED BY LEARNER]


## Unfilled ADR

Author the V10 memory/routing/fallback policy in `missions/M39/adr_prompt.md`
using `templates/ADR.md`. Persistence, provenance/retention, retrieval scope,
route precedence, retry/fallback bounds, degraded outputs, and circuit
breaking are the decision. Do not implement an M40 eval harness in the ADR.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]


## M38 → M39 → M40 handoff

M40 receives declared memory, route, fallback, and trace surfaces
(`retrieved_ids`, excluded reasons, `used_memory_ids`, route name, attempt
count, circuit state, degraded flag, provenance/expiry). It may evaluate them.
It may not pretend this teaching fixture is a production runtime.

M39 does not open RAG, Qdrant, sampling labs, or an eval harness.
This wrapper is **not a production** agent runtime.


## Mission summary

You classified M38 fields, retrieved with provenance, routed with predicates,
wrapped M38, degraded instead of lying, and bounded a failing ladder.
Controlled failures showed why expiry and circuits are policy, not prompts.

Next mission: evaluate the system systematically. Not here.


In [ ]:
assert whole.route == "catalog_purchase" and whole.terminal == "complete"
assert whole.degraded is False and whole.attempts == 1
assert whole.posted_amount == CATALOG_PRICE == M38_CATALOG_PRICE
assert whole.effect_count == 1
assert whole.workflow.session.executions == ["lookup_catalog_price", "post_ledger_entry"]
assert relevance.retrieved_ids == (MEM_SKU7_PRICE,)
assert MEM_SKU7_STALE not in stale.retrieved_ids
assert stale.posted_amount == M38_CATALOG_PRICE
assert lookup_run.route == "catalog_lookup" and lookup_run.effect_count == 0
assert refused.route == "no_match" and refused.workflow is None
assert primary_fail.terminal == "degraded" and primary_fail.degraded is True
assert primary_fail.effect_count == 0
assert looped.terminal == "circuit_open" and looped.circuit_open is True
assert looped.attempts <= MAX_ATTEMPTS
assert broken_stale.posted_amount != M38_CATALOG_PRICE
assert broken_stale.terminal == "complete" and broken_stale.degraded is False
assert repaired_stale.posted_amount == M38_CATALOG_PRICE
assert broken_stale.posted_amount != repaired_stale.posted_amount
assert broken_osc.attempts > MAX_ATTEMPTS and broken_osc.circuit_open is False
assert repaired_osc.terminal == "circuit_open" and repaired_osc.circuit_open is True
assert type(whole.workflow.state).__module__ == "missions.M38.agent_workflow"
assert whole.workflow.state.inference["weights_updated"] is False
print("M39 integrity checks passed")
print(handoff_contract()["handoff"])
